<center><span style="font-size:50px;"><b>Abe-Suzuki Earthquake Network - 5 km</b></span></center>

---

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Abe-Suzuki Earthquake Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_ABE.py
Convert to notebook: python convert_to_notebook.py ITALY_network_ABE.py notebooks/ITALY_network_ABE.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_abe_suzuki_network
from src.metrics import (
    estimate_gamma_mle,
    test_power_law,
    verify_balanced_degrees,
)
from src.assortativity import (
    attach_catalog_attrs,
    compute_assortativity,
    plot_assortativity,
    plot_knn,
)
from src.robustness import simulate_robustness, plot_robustness_curves
from src.centrality import (
    compute_all_centralities,
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_directed_louvain,
    run_hdbscan_geo,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    compare_partitions,
    plot_partition_scores,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    analyze_in_out_degree_distribution,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)
CUT_YEAR    = 1985
TARGET_CRS  = "epsg:32632"  # UTM Zone 32N — appropriate for Italy
SAVE_PDF: bool = True    # save vector PDF for report
SAVE_JPG: bool = True    # save raster JPG for slides/preview

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "abe")

## Data Loading

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

_ITALY_POLYGON = Polygon([
    (13.5, 44.5),  # Northeast (Adriatic Sea)
    (19.0, 41.2),  # Southeast (Apulia Sea)
    (19.0, 35.6),  # Sea between Italy and Libya
    (11.7, 35.5),  # Sea east of Tunisia
    (11.6, 37.9),  # West of Sicily
    (7.2,  38.0),  # Southwest of Sardinia
    (7.2,  42.6),  # West of Corsica
    (5.2,  45.5),  # Lyon
    (11.6, 48.4),  # Munich
    (16.0, 46.7),  # Between Austria and Slovenia
    (13.5, 44.5),  # closing vertex
])
df_net = df_net[df_net.apply(
    lambda r: _ITALY_POLYGON.contains(Point(r["longitude"], r["latitude"])), axis=1
)].reset_index(drop=True)
print(f"After polygon filter: {len(df_net):,} earthquakes remain")

# df_net = df_net[
#     (df_net["magnitude"] > 2.0) &
#     (df_net["magnitude"] <= 10.0) &
#     (df_net["time"].dt.year >= 2005) &
#     (df_net["time"].dt.year <= 2015)
# ].reset_index(drop=True)
# print(f"After magnitude [2–10] and year [2005–2015] filter: {len(df_net):,} earthquakes remain")

neg = df_net[df_net["depth_km"] < 0]
print(f"\nNegative-depth events: {len(neg)} — kept, collapse to cell_z = -1")

## Network Construction

In [ ]:
G_5km = build_abe_suzuki_network(df_net, cell_size_km=5, target_crs=TARGET_CRS)

print(f"\n5 km network: {G_5km.number_of_nodes():,} nodes  {G_5km.number_of_edges():,} edges")

## Hub Map — 2D

The top-2% highest-degree cells are mapped geographically to locate the
most seismically active spatial clusters. Degree here is total unweighted
degree, so hubs correspond to cells visited most frequently in the
earthquake sequence — the busiest fault segments.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "cell_id": n,
        "degree":  G_5km.degree(n),
        "lat":     G_5km.nodes[n]["lat"],
        "lon":     G_5km.nodes[n]["lon"],
    }
    for n in G_5km.nodes() if "lat" in G_5km.nodes[n]
])
df_nodes  = df_nodes[df_nodes["degree"] > 0]
threshold = df_nodes["degree"].quantile(0.98)
df_hubs   = df_nodes[df_nodes["degree"] >= threshold].copy()
print(f"Mapping top 2% hubs: {len(df_hubs)} cells (degree >= {threshold:.0f})")

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="degree", size="degree",
    color_continuous_scale="plasma",
    map_style="carto-positron",
    hover_name="cell_id",
    title="Abe-Suzuki Network Hubs — Italy 5×5×5 km (top 2% by degree)",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0}, width=MAP_WIDTH, height=MAP_HEIGHT,
                  coloraxis_colorbar=dict(title="Degree"),
                  map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS))
save_plotly(fig, "hub_map_2d_italy_5km")
fig.show()

## Network Node Map — Degree & Depth

Every network node (5×5×5 km spatial cell) is placed on an Italy basemap.
**Marker size** encodes total degree — the number of distinct transition events
entering or leaving the cell — so the largest circles are the most seismically
active cells in the sequence. **Colour** encodes cell-centre depth (km), from
shallow crustal seismicity along the Apennine belt (yellow) to deeper
subduction-related seismicity beneath the Tyrrhenian back-arc (purple).
Together, size and colour reveal that most network hubs are shallow
(seismogenic crust, 5–20 km), while deep cells are relatively isolated —
consistent with the Chiarabba *et al.* (2005) Italy seismicity model.

In [ ]:
_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

df_nodes = pd.DataFrame([
    {
        "cell_id":  n,
        "degree":   G_5km.degree(n),
        "lat":      G_5km.nodes[n]["lat"],
        "lon":      G_5km.nodes[n]["lon"],
        "depth_km": (int(n.split("_")[2]) + 0.5) * 5.0,
    }
    for n in G_5km.nodes() if "lat" in G_5km.nodes[n]
])
df_nodes = df_nodes[df_nodes["degree"] > 0].copy()
df_nodes["depth_km"] = df_nodes["depth_km"].clip(upper=200)
df_nodes = df_nodes.sort_values("depth_km", ascending=False)  # deep-first → shallow on top

print(f"Node map: {len(df_nodes):,} active cells  "
      f"degree range [{df_nodes['degree'].min()}–{df_nodes['degree'].max()}]  "
      f"depth range [{df_nodes['depth_km'].min():.0f}–{df_nodes['depth_km'].max():.0f}] km")

fig = px.scatter_map(
    df_nodes,
    lat="lat", lon="lon",
    size="degree",
    color="depth_km",
    color_continuous_scale="plasma_r",
    size_max=30,
    map_style="carto-positron",
    hover_name="cell_id",
    hover_data={"degree": True, "depth_km": ":.1f", "lat": ":.3f", "lon": ":.3f"},
    labels={"depth_km": "Depth (km)", "degree": "Degree"},
    title="Abe-Suzuki Network — Italy 5×5×5 km (size = degree, colour = depth)",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    coloraxis_colorbar=dict(title="Depth (km)"),
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "node_map_degree_depth_italy_5km")
fig.show()

---

---

---

<center><span style="font-size:50px;"><b>DEGREE DISTRIBUTION</b></span></center>

## Degree Balance Verification

By construction, every interior node in the Abe-Suzuki network must satisfy
$k_i^{\text{in}} = k_i^{\text{out}}$ (the cell that received the $n$-th event
also sent the $(n+1)$-th). Only the first and last events in the catalog
produce a single unbalanced node each.

A non-empty list here signals a data-loading or sorting error — the catalog
must be strictly ordered by time before network construction.

In [ ]:
print("Checking degree balance (interior nodes: in-strength == out-strength)...")
result = verify_balanced_degrees(G_5km)
if isinstance(result, list):
    print(f"Unbalanced nodes: {result}")

## Degree Distribution — Linear Scale

Linear-scale histogram of node degrees, truncated at $k = 50$ to reveal the
low-degree bulk of the distribution. In a scale-free network this range is
dominated by nodes with very few connections; the exponential-like drop-off
here contrasts with the heavy tail visible on the log-log plots.

In [ ]:
plot_degree_distribution_linear(G_5km, "Abe-Suzuki Italy (5 km)", max_degree=50)

## Degree Distribution — Log-Log Scatter

Log-log scatter of $P(k)$ vs $k$ is the canonical diagnostic for a power-law
tail: a straight line with slope $-\gamma$ on this plot implies $P(k) \propto k^{-\gamma}$.
Plotting in-degree and out-degree separately tests whether the directed
asymmetry (if any) modifies the exponent — for earthquake networks with
balanced interior degrees the two should be nearly identical.

In [ ]:
analyze_in_out_degree_distribution(G_5km, "Abe-Suzuki Italy (5 km)")

analyze_degree_distribution(G_5km, "Abe-Suzuki Italy (5x5x5 km)")

## MLE Gamma Estimation

The maximum-likelihood estimator for the power-law exponent (Clauset et al. 2009):
$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$
where $n$ is the number of degrees $\geq k_{\min}$ and $k_{\min} = 10$.

The Clauset-Shalizi-Newman (CSN) likelihood-ratio test compares the power-law
to an exponential alternative: $R > 0$ with $p < 0.05$ rejects the exponential
and supports the power-law. Abe & Suzuki (2004) found $\gamma \approx 2.2$
for the Japan catalog; $\gamma < 3$ implies a divergent second moment and
enhanced vulnerability to targeted attack.

The MLE $\hat{\gamma}$ is used as the fixed slope in the log-binned and CCDF
plots below, with only the amplitude fitted to the data — giving a visually
honest overlay that does not double-count estimation error.

In [ ]:
degs_5km = [d for _, d in G_5km.degree(weight="weight") if d > 0]
gamma_5km = estimate_gamma_mle(degs_5km, k_min=10)
print(f"MLE γ (Italy 5 km network): {gamma_5km:.3f}")

result = test_power_law(degs_5km, k_min=10)
print(f"  CSN test: γ={result['gamma']} (σ={result['sigma']})  "
      f"R={result['R']}  p={result['p_value']}  → {result['verdict']}")

## Degree Distribution — Log Binning

Logarithmic binning reduces noise in the sparse high-$k$ tail by grouping
degrees into exponentially spaced bins and normalising by bin width.
This is the standard method recommended by Clauset et al. (2009) for
visually assessing power-law fit quality. The overlaid line uses the MLE
exponent $\hat{\gamma}$ with amplitude fitted to the tail ($k \geq 10$).

In [ ]:
analyze_degree_distribution_log_binning(G_5km, "Abe-Suzuki Italy (5x5x5 km)",
                                         k_min_fit=10, gamma_mle=gamma_5km)

## Degree Distribution — CCDF

The complementary cumulative distribution function $P(K \geq k)$ avoids
binning artefacts entirely and is monotone non-increasing by construction.
For a power-law $P(k) \propto k^{-\gamma}$, the CCDF follows
$P(K \geq k) \propto k^{-(\gamma-1)}$; the overlaid line uses the MLE
exponent so the slope is $-(\hat{\gamma}-1)$.

In [ ]:
plot_ccdf_with_fit(G_5km, "Abe-Suzuki Italy 5 km", k_min_fit=10, gamma_mle=gamma_5km)

---

---

---

<center><span style="font-size:50px;"><b>MACROSCOPIC METRICS</b></span></center>

The small-world signature (Watts & Strogatz 1998) requires simultaneously
$C_{\text{real}} \gg C_{\text{ER}}$ (high local clustering) and
$L_{\text{real}} \approx L_{\text{ER}}$ (short average path length),
where $C_{\text{ER}} \approx \langle k \rangle / N$ and $L_{\text{ER}} \approx \ln N / \ln \langle k \rangle$.

**Giant component fraction** measures network integrity: values $> 0.9$ indicate
a well-connected seismic system; a fragmented network would imply spatially
isolated fault clusters with no topological path between them.
The adjacency matrix sparsity plot visually confirms the ultra-sparse structure
characteristic of real-world networks with $\langle k \rangle \ll N$.

In [ ]:
G = G_5km

print("--- Giant Component ---")
wcc     = list(nx.weakly_connected_components(G))
G_giant = G.subgraph(max(wcc, key=len)).copy()
pct     = G_giant.number_of_nodes() / G.number_of_nodes() * 100
print(f"Components: {len(wcc)} | Giant: {G_giant.number_of_nodes():,} nodes ({pct:.1f}%)")

print("\n--- Clustering Coefficient ---")
G_und = G_giant.to_undirected()
G_und.remove_edges_from(nx.selfloop_edges(G_und))
avg_c = nx.average_clustering(G_und)
p_rnd = (2 * G_und.number_of_edges()
         / (G_giant.number_of_nodes() * (G_giant.number_of_nodes() - 1)))
print(f"Avg clustering C = {avg_c:.4f}  (Erdos-Renyi expected: {p_rnd:.4f})")

print("\n--- Average Path Length & Diameter ---")
t0       = time.time()
avg_L    = nx.average_shortest_path_length(G_und)
diameter = nx.diameter(G_und)
n_giant  = G_giant.number_of_nodes()
mean_k   = 2 * G_und.number_of_edges() / n_giant
L_er     = np.log(n_giant) / np.log(mean_k)
print(f"L_real = {avg_L:.2f}  |  L_ER ≈ ln N / ln ⟨k⟩ = {L_er:.2f}  |  Diameter = {diameter}  ({time.time()-t0:.0f}s)")

print("\n--- Adjacency Matrix Sparsity ---")
adj      = nx.to_scipy_sparse_array(G)
sparsity = 1.0 - adj.nnz / G.number_of_nodes()**2
print(f"Shape: {adj.shape}  |  Non-zeros: {adj.nnz:,}  |  Sparsity: {sparsity*100:.4f}%")

plt.figure(figsize=(8, 8))
plt.spy(adj, markersize=0.05, color="darkblue")
plt.title("Adjacency Matrix (5 km Italy network)", fontsize=14)
plt.xlabel("Target Cell Index", fontsize=12)
plt.ylabel("Source Cell Index", fontsize=12)
plt.legend(handles=[
    mpatches.Patch(facecolor="darkblue", label="Link (edge present)"),
    mpatches.Patch(facecolor="white", edgecolor="lightgray", label="No link"),
], loc="upper right", fontsize=9, framealpha=0.8)
plt.tight_layout()
savefig("adjacency_matrix_italy_5km")
plt.show()

---

---

---

<center><span style="font-size:50px;"><b>CENTRALITY</b></span></center>

## Centrality Analysis — 5 Measures

Five complementary centrality measures capture distinct structural roles in
the Abe–Suzuki seismic cell network.  Each measure encodes a different aspect of
network influence; comparing them reveals whether prominent nodes are prominent
for the same reason (redundancy) or for different reasons (orthogonality).

### Degree

**Total degree** $k_i = k^{\mathrm{in}}_i + k^{\mathrm{out}}_i$ identifies the
most globally active cells, regardless of the direction of influence.

### Path-based measures

**PageRank** (Page *et al.* 1998) solves the stationary equation
$\mathbf{r} = \alpha A^T D^{-1} \mathbf{r} + (1-\alpha)\mathbf{1}/N$
(damping factor $\alpha = 0.85$).  High PageRank indicates a *stress sink*: a cell
that persistently receives seismic flow from well-connected predecessors, not merely
from many predecessors.

**Closeness centrality** $C_i = (N-1)/\sum_j d_{ij}$ is the inverse mean
shortest-path distance.  High-closeness cells can propagate stress signals fastest
across the network — seismological analogue of rapid aftershock diffusion.

**Betweenness centrality** $B_i = \sum_{s \neq i \neq t} \sigma_{st}(i)/\sigma_{st}$
(Freeman 1977) counts the fraction of shortest paths that pass through $i$.
High-betweenness cells act as *seismic bridges* between distinct fault clusters;
removing them would fragment the network into isolated communities.

### Local topology

**Clustering coefficient** $C_i = \frac{2 t_i}{k_i(k_i-1)}$ (Watts & Strogatz 1998)
measures the probability that two neighbours of cell $i$ are also neighbours of each
other.  High clustering signals fault junctions or dense aftershock clusters where
multiple fault strands intersect in a small spatial cell.

### Interpretation of the correlation heatmap

Pairs of measures with Spearman $\rho > 0.9$ are functionally redundant and can
be represented by a single measure in downstream analysis.  Pairs with
$\rho < 0.3$ are structurally orthogonal and reveal genuinely different node roles.
Expected orthogonalities: Betweenness vs Degree (bridges may have low degree),
Clustering vs Degree (peripheral low-degree junctions can have high clustering)

In [ ]:
print("Computing centrality measures (5 km network)...")
df_centrality = compute_all_centralities(G_5km, k_betweenness=1000, seed=42)

for metric, label in [
    ("Degree",      "Most Active Swarms"),
    ("PageRank",    "Stress Sinks"),
    ("Closeness",   "Topological Centers"),
    ("Betweenness", "Fault Bridges"),
    ("Clustering",  "Fault Junctions (Clustering)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "Abe-Suzuki Italy (5 km)")
plot_top_n_cells(df_centrality, top_n=10, title="Abe-Suzuki Italy (5 km)")

## Centrality Geo Maps

Geographic projection of the top-10 nodes per centrality metric onto the
Italy basemap. The **convergence map** (colour = count of metrics for which a
node ranks in the top-10) identifies structurally robust hubs: cells that are
simultaneously active, well-connected, and located on critical fault bridges.

Seismological expectation: hubs should cluster along the Apennine arc and
in historically active regions (L'Aquila, Amatrice, Irpinia); divergence
between metrics reveals nodes that are locally active but globally peripheral.

In [ ]:
# Interactive map: dropdown to switch metric, top-10 nodes on Italy basemap.
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
# Overlap map: colour = number of metrics for which a node is in the top-10.
# Multi-metric hubs (warm colours) identify the most structurally critical cells.
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

---

---

---

<center><span style="font-size:50px;"><b>COMMUNITY DETECTION</b></span></center>

## Community Detection — Five Algorithms

The Abe-Suzuki network is a **directed transition network** (Markov chain on spatial
cells). Conceptually, communities are sets of cells that form quasi-closed seismic flow
basins — once a sequence enters a basin, it tends to stay there.  Two classes of method
are applied: **flow-based** (InfoMap, run on the directed graph — conceptually correct
for this network type) and **structural** (Louvain, Consensus, Spectral — run on the
undirected symmetrised graph, finding dense subgraphs regardless of flow direction).
HDBSCAN-Geographic serves as a pure spatial baseline with no network information.

**InfoMap** (Rosvall & Bergstrom 2008) is the primary method for directed transition
networks.  It minimises the map equation
$$L(\mathcal{M}) = q_\curvearrowleft H(\mathcal{Q}) + \sum_{i=1}^{m} p_i^\circlearrowleft H(\mathcal{P}^i),$$
which balances the cost of describing inter- vs intra-community movement in the directed
random walk.  Run here on the directed giant component — no undirected conversion.

**Louvain** (Blondel *et al.* 2008, via Leiden algorithm) maximises the
Reichardt–Bornholdt modularity $Q_\gamma$ on the undirected symmetrised graph
(resolution $\gamma=0.5$ to obtain $\mathcal{O}(5\text{–}10)$ communities).
Structural communities here correspond to cells that frequently co-occur in
the same aftershock cluster — not necessarily geographic provinces.

**Consensus Louvain** (Lancichinetti & Fortunato 2012): 100 Louvain runs →
co-occurrence matrix $C_{ij}$ → final Louvain on $C$; eliminates single-run noise.

**Spectral** (Jordan & Weiss 2002): $k$-means on the $k$ smallest eigenvectors of
the normalised Laplacian; $k$ set to the InfoMap community count.

**HDBSCAN-Geographic**: density clustering on $(x,y)$ projected coordinates only;
no graph structure — pure spatial baseline.

All five partitions are scored on 9 quality metrics and compared pairwise via NMI
(5×5 matrix).  Agreement between InfoMap and Louvain is the key diagnostic: high
NMI implies flow basins ≈ dense structural clusters; low NMI implies directed flow
organises the network differently from undirected modularity.

In [ ]:
print("Building undirected giant component for structural community detection...")
G_und_giant = G_giant.to_undirected()
G_und_giant.remove_edges_from(nx.selfloop_edges(G_und_giant))

# ── InfoMap on the DIRECTED graph (primary method for this network type) ──────
print("InfoMap (directed)...")
community_map_infomap = run_infomap(G_giant, directed=True, seed=42)
k_infomap = len(set(community_map_infomap.values()))
print(f"  {k_infomap} communities")
plot_community_geo(
    G_und_giant, community_map_infomap,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap",
)

# ── Louvain on undirected (structural comparison, γ=0.5) ─────────────────────
print("Louvain (undirected, γ=0.5)...")
community_map = run_louvain(G_und_giant, seed=42, resolution=0.5)
k_louvain = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G_und_giant, community_map,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

# ── Consensus Louvain (100 runs, γ=0.5) ─────────────────────────────────────
print("Consensus Louvain (100 runs, γ=0.5)...")
community_map_consensus = run_consensus_louvain(G_und_giant, n_runs=100, seed=42, resolution=0.5)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G_und_giant, community_map_consensus,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

# ── Spectral clustering (k = InfoMap count for consistency) ───────────────────
print(f"Spectral clustering (k={k_infomap})...")
community_map_spectral = run_spectral(G_und_giant, k=k_infomap, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities")
plot_community_geo(
    G_und_giant, community_map_spectral,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Spectral",
)

# ── HDBSCAN on geographic coordinates (spatial baseline) ─────────────────────
print("HDBSCAN-Geographic...")
community_map_hdbscan_geo = run_hdbscan_geo(G_und_giant, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} communities")
plot_community_geo(
    G_und_giant, community_map_hdbscan_geo,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=50, method_name="HDBSCAN-Geographic",
)

# ── NMI comparison heatmap (5 × 5) ───────────────────────────────────────────
print("Computing NMI across methods...")
partitions = {
    "Louvain":       community_map,
    "Consensus":     community_map_consensus,
    "Spectral":      community_map_spectral,
    "InfoMap":       community_map_infomap,
    "HDBSCAN-Geo":   community_map_hdbscan_geo,
}
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="Abe-Suzuki Italy (5 km)")

scores_df = compare_partitions(G_und_giant, partitions, cell_size_km=5.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="Abe-Suzuki Italy (5 km)")

## Adjacency Matrix — Community Order

Reordering nodes by InfoMap community reveals the **block structure** hidden in the
raw adjacency matrix (where nodes are ordered arbitrarily by cell ID). Within each
community nodes are sorted by degree descending, so **hub nodes appear at the leading
edge of each block** — the dense rows/columns at block boundaries.

Left panel: full reordered matrix downsampled to ≈800 px (each pixel = sum of a
$\sim$20×20 node block). Community boundaries are marked with white lines.
Right panel: $k \times k$ block-density heatmap where entry $(i,j)$ is the fraction
of possible edges between communities $i$ and $j$ that actually exist. High diagonal
values indicate tight intra-community structure; off-diagonal density reveals the
inter-community coupling responsible for the small-world behaviour.

In [ ]:
import scipy.sparse as sp

# ── Reorder nodes: community first, then by degree descending (hubs at block top) ──
_comm_ids_sorted = sorted(set(community_map_infomap.values()))
_node_order = sorted(
    G_und_giant.nodes(),
    key=lambda n: (_comm_ids_sorted.index(community_map_infomap.get(n, 0)),
                   -G_und_giant.degree(n))
)
_N = len(_node_order)
_n2i = {n: i for i, n in enumerate(_node_order)}

# Build reordered sparse adjacency
_r, _c = [], []
for u, v in G_und_giant.edges():
    if u in _n2i and v in _n2i:
        _r += [_n2i[u], _n2i[v]]; _c += [_n2i[v], _n2i[u]]
_A = sp.csr_matrix((np.ones(len(_r), dtype=np.float32), (_r, _c)), shape=(_N, _N))

# Downsample to ≈800 px
_bs = max(1, _N // 800)
_nb = _N // _bs
_Ads = np.zeros((_nb, _nb), dtype=np.float32)
for _bi in range(_nb):
    _Ads[_bi] = np.asarray(
        _A[_bi * _bs: (_bi + 1) * _bs].sum(axis=0)
    ).ravel()[: _nb * _bs].reshape(_nb, _bs).sum(axis=1)

# Community boundary positions in the downsampled matrix
_comm_sizes = [sum(1 for n in _node_order if community_map_infomap.get(n) == c)
               for c in _comm_ids_sorted]
_boundaries = np.cumsum(_comm_sizes)[:-1] / _bs  # in downsampled coordinates

# ── Block density matrix ──────────────────────────────────────────────────────
_nc = len(_comm_ids_sorted)
_ci = {c: i for i, c in enumerate(_comm_ids_sorted)}
_B = np.zeros((_nc, _nc))
for u, v in G_und_giant.edges():
    i = _ci.get(community_map_infomap.get(u, -1), 0)
    j = _ci.get(community_map_infomap.get(v, -1), 0)
    _B[i, j] += 1
    if i != j:
        _B[j, i] += 1
for i in range(_nc):
    for j in range(_nc):
        s_i, s_j = _comm_sizes[i], _comm_sizes[j]
        denom = s_i * s_j if i != j else s_i * (s_i - 1) / 2
        if denom > 0:
            _B[i, j] /= denom

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
ax.imshow(np.log1p(_Ads), aspect="auto", cmap="Blues", interpolation="nearest")
for bnd in _boundaries:
    ax.axhline(bnd, color="white", lw=0.6, alpha=0.8)
    ax.axvline(bnd, color="white", lw=0.6, alpha=0.8)
ax.set_title("Reordered Adjacency Matrix\n(nodes sorted by community, then by degree)",
             fontsize=12)
ax.set_xlabel("Target Cell (community order)", fontsize=10)
ax.set_ylabel("Source Cell (community order)", fontsize=10)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.legend(handles=[
    mpatches.Patch(facecolor=plt.cm.Blues(0.85), label="Links (darker = more)"),
    mpatches.Patch(facecolor="white", edgecolor="lightgray", label="No links"),
], loc="lower right", fontsize=8, framealpha=0.8)

im = axes[1].imshow(np.log1p(_B * 1000), cmap="YlOrRd", aspect="auto")
axes[1].set_title(f"Community Block-Density Matrix\n({_nc} Louvain communities)",
                  fontsize=12)
axes[1].set_xlabel("Community j", fontsize=10)
axes[1].set_ylabel("Community i", fontsize=10)
plt.colorbar(im, ax=axes[1], label="log(1 + density × 1000)")

fig.suptitle("Adjacency Structure by Community — Italy 5 km", fontsize=14, y=1.01)
plt.tight_layout()
savefig("adjacency_matrix_community_order_italy")
plt.show()

## Directed Community Detection

Directed modularity $Q_d = \frac{1}{m}\sum_{ij}\left[A_{ij} - \frac{k_i^{\text{out}}k_j^{\text{in}}}{m}\right]\delta(c_i,c_j)$
(Leicht & Newman 2008) groups nodes that send *and* receive seismic activity
within the same community, going beyond edge density alone.
Implemented via the Leiden algorithm (Traag et al. 2019) for better
convergence guarantees than vanilla Louvain.

**Interpretation of NMI vs undirected Louvain**: high NMI means fault-zone
communities are direction-agnostic (stress flows symmetrically within zones);
low NMI reveals directional flow communities — e.g., a slab that consistently
triggers shallow crustal events but never the reverse.

In [ ]:
# Standard Louvain operates on the undirected projection and maximises
# edge density.  Directed Louvain (Leiden algorithm, Leicht-Newman 2008
# modularity) accounts for asymmetric flow: two nodes belong to the same
# community only if they preferentially *send and receive* seismic activity
# within the group.  NMI against undirected Louvain quantifies how much
# directionality changes the community structure.

print("Running directed Louvain (Leiden algorithm) on G_5km...")
community_map_directed = run_directed_louvain(G_5km, seed=42)
k_directed = len(set(community_map_directed.values()))
print(f"  {k_directed} directed communities  (vs {k_louvain} undirected)")

plot_community_geo(
    G_5km, community_map_directed,
    title="Abe-Suzuki Italy (5 km)",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Directed Louvain (Leiden)",
)

# Compare directed vs all undirected methods
partitions_full = {**partitions, "Directed Louvain": community_map_directed}
nmi_full = compute_nmi_matrix(partitions_full)
display(nmi_full.round(3))
plot_nmi_heatmap(nmi_full, title="Abe-Suzuki Italy — directed vs undirected")

---

---

---

<center><span style="font-size:50px;"><b>ASSORTATIVITY</b></span></center>

## Assortativity Analysis — Six Diagnostics

Assortativity measures the tendency of nodes to connect to similar nodes.
For earthquake networks it answers: do seismically active cells trigger other
active cells, or do they discharge into quiet peripheral ones?

### Newman assortativity coefficient $r$

The scalar summary of degree-degree or attribute-attribute mixing is
$$r = \frac{\sum_{ij}(A_{ij} - k_i k_j / 2m)\,x_i x_j}
{\sum_{ij}(k_i\delta_{ij} - k_i k_j / 2m)\,x_i^2},$$
where $x_i$ is the node attribute (degree, mean depth, mean magnitude) and
$m$ is the number of edges (Newman 2002).  $r \in [-1, 1]$: positive values
indicate assortative mixing (similar connects to similar); negative values
indicate disassortative (hub-and-spoke) structure.

**Degree assortativity** $r < 0$ (disassortative) is the canonical scale-free
signature: mainshock hubs discharge into many low-degree aftershock cells rather
than into other hubs.  This is structurally equivalent to the star-like topology
of Omori aftershock trees.

**Depth assortativity** tests whether seismic sequences respect depth horizons.
A positive $r$ would indicate that crustal events trigger other crustal events
(shallow-to-shallow coupling), consistent with distinct seismogenic depth
horizons (upper crust, lower crust, Moho transition).  $r \approx 0$ suggests
free cross-depth coupling, as expected for a thick seismogenic zone with no
strong layering.

**Magnitude assortativity** tests whether high-magnitude zones cluster together
in the temporal sequence — evidence of mainshock–mainshock triggering rather
than mainshock-to-aftershock sequences.

### Degree-mixing exponent $\mu$

The average nearest-neighbour degree $\bar{k}_{\mathrm{nn}}(k)$ vs $k$ on
log-log axes gives the **degree-mixing exponent**
$$\bar{k}_{\mathrm{nn}}(k) \sim k^\mu, \qquad \mu < 0 \text{ (disassortative)},$$
(Pastor-Satorras *et al.* 2001).  Unlike the scalar $r$, $\mu$ characterises
the degree-degree correlation across the full degree spectrum: $\mu = -1$ is
pure hub-and-spoke, $\mu = 0$ is uncorrelated, $\mu = +1$ is perfect assortativity.

### Directed degree mixing

Foster *et al.* (2010) generalise assortativity to directed networks by
decomposing edge correlations into four types based on endpoint degree type:
out→out, out→in, in→out, in→in.  In earthquake networks:
- **out→out**: do prolific-triggering cells preferentially trigger other prolific cells?
- **out→in**: do triggering cells connect to cells that are themselves heavily triggered?
The Pearson $r$ of each panel is the directed analogue of Newman assortativity.

### Rich-club coefficient $\phi_{\mathrm{norm}}(k)$

$$\phi(k) = \frac{2\,E_{>k}}{N_{>k}(N_{>k}-1)}, \qquad
\phi_{\mathrm{norm}}(k) = \frac{\phi(k)}{\langle\phi_{\mathrm{rand}}(k)\rangle},$$
where $E_{>k}$ is the number of edges among the $N_{>k}$ nodes with degree above
threshold $k$ (Colizza *et al.* 2006).  $\phi_{\mathrm{norm}} < 1$ across all $k$
confirms rich-club absence: hubs in the earthquake network form a hub–periphery
structure rather than a densely connected elite — consistent with aftershock trees.

### Depth E-I index

The **E-I index** (Krackhardt & Stern 1988) measures the balance between
cross-group (External) and within-group (Internal) edges:
$$\text{E-I} = \frac{E - I}{E + I} \in [-1, 1],$$
where depth layers are defined as shallow ($d \leq 15$ km), intermediate
($15 < d \leq 35$ km) and deep ($d > 35$ km).

$\text{E-I} < 0$ indicates **homophily**: cells predominantly trigger other
cells in the same depth layer — distinct seismogenic horizons with little
cross-depth energy transfer.  $\text{E-I} > 0$ indicates **heterophily**:
triggering crosses depth layers, consistent with a throughgoing fault zone
or deep-to-shallow stress transfer (e.g. deep interplate → shallow intraplate).
For the Italian catalog, the presence of both shallow crustal seismicity and
deeper subduction-related events (Calabrian arc) makes this a sensitive test
of tectonic segmentation.

In [ ]:
print("Attaching catalog attributes to nodes...")
attach_catalog_attrs(G_5km, df_net, cell_size_km=5.0, target_crs=TARGET_CRS)

print("Computing assortativity measures...")
df_assort = compute_assortativity(G_5km)
display(df_assort)

print("\nMixing pattern plots:")
plot_assortativity(G_5km, title="Abe-Suzuki Italy (5 km)")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G_5km, title="Abe-Suzuki Italy (5 km)", gamma=gamma_5km)

---

---

---

<center><span style="font-size:50px;"><b>ROBUSTNESS</b></span></center>

Albert et al. (2000) showed that scale-free networks exhibit a sharp
dichotomy: **robust** to random node failure (the giant connected component
survives removal of many low-degree nodes) but **fragile** to targeted attack
on hubs (GCC collapses after removing a small fraction of the highest-degree nodes).

The critical fraction $f_c$ (at which GCC $< 0.05$) under targeted attack
is the key metric: lower $f_c$ → more scale-free fragility.
Physical implication: monitoring a handful of high-degree seismic cells
(e.g., the top-2% hubs) provides disproportionate forecasting leverage.

In [ ]:
n_gcc = G_und_giant.number_of_nodes()
m_gcc = G_und_giant.number_of_edges()
G_er_rob = nx.gnm_random_graph(n_gcc, m_gcc, seed=42)

print("Simulating robustness...")
rob_results = {
    "Random (Real)":   simulate_robustness(G_und_giant, strategy="random",   seed=42),
    "Targeted (Real)": simulate_robustness(G_und_giant, strategy="targeted"),
    "Random (ER)":     simulate_robustness(G_er_rob,    strategy="random",   seed=42),
    "Targeted (ER)":   simulate_robustness(G_er_rob,    strategy="targeted"),
}
plot_robustness_curves(rob_results, title="Abe-Suzuki Italy (5 km)")

---

---

---

# Export Results

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_eq_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved to {out_path}  ({len(df_final):,} rows)")

# Cache network graphs as pickle for fast loading in cross_catalog_comparison.
# Set USE_CACHED_NETWORKS = True there to skip rebuilding (~minutes → seconds).
for fname, G in [("italy_G5km.pkl", G_5km)]:
    pkl_path = RESULTS_DIR / "cache" / fname
    with open(pkl_path, "wb") as f:
        pickle.dump(G, f)
    print(f"Network cached → {pkl_path}")

# Gephi-compatible export (GEXF preserves node/edge attributes).
# Open in Gephi → Layout → ForceAtlas2 for publication-quality figures.
gexf_path = RESULTS_DIR / "gephi" / "italy_G5km.gexf"
nx.write_gexf(G_5km, gexf_path)
print(f"Gephi export → {gexf_path}")